# ESPAC Crop LCI -> XML Generator

Generate one XML file per row from the crop LCI pipeline using `generalComment` in the template exchanges as field keys.


In [ ]:
from pathlib import Path
import re
import json
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from datetime import date
from IPython.display import display, Markdown
from scripts.crop_groups import canonical_crop_group_token
import yaml

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent

LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_filtered_export_summary.json'

if not LATEST_FILTERED_SUMMARY_META_PATH.exists():
    raise RuntimeError(
        f"Missing metadata file: {LATEST_FILTERED_SUMMARY_META_PATH}. "
        "Select filters/strategy in notebook 2 and export filtered CSV first."
    )

try:
    summary_meta = json.loads(LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
except Exception as exc:
    raise RuntimeError(f"Failed reading strategy metadata {LATEST_FILTERED_SUMMARY_META_PATH}: {exc}") from exc

SUMMARY_STRATEGY = str(summary_meta.get('summary_level', '')).strip().lower()
ALLOWED_SUMMARY_STRATEGIES = {
    'province', 'region', 'crop_national', 'cropping_system',
    'irrig_m3_class', 'farm_size_class', 'crop_group', 'crop_group_national',
}
if not SUMMARY_STRATEGY:
    raise RuntimeError(
        f"Strategy is missing in metadata file {LATEST_FILTERED_SUMMARY_META_PATH}. "
        "Set Summary in notebook 2 and export filtered CSV before XML generation."
    )
if SUMMARY_STRATEGY not in ALLOWED_SUMMARY_STRATEGIES:
    raise RuntimeError(
        f"Unsupported summary strategy '{SUMMARY_STRATEGY}' in {LATEST_FILTERED_SUMMARY_META_PATH}. "
        f"Allowed values: {sorted(ALLOWED_SUMMARY_STRATEGIES)}"
    )

SUMMARY_SOURCE = 'metadata'
SUMMARY_TOKEN = SUMMARY_STRATEGY

DEFAULT_TEMPLATE = PROJECT_DIR / "inputs/crops00001.XML"
FILTERED_LCI_FROM_META = Path(str(summary_meta.get('filtered_csv', '')).strip())
if not FILTERED_LCI_FROM_META.exists():
    raise RuntimeError(
        f"Filtered CSV from metadata does not exist: {FILTERED_LCI_FROM_META}. "
        "Run notebook 2 export again before XML generation."
    )

DEFAULT_LCI = PROJECT_DIR / f"outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_{SUMMARY_TOKEN}.csv"
if DEFAULT_LCI.exists():
    if DEFAULT_LCI.stat().st_mtime < FILTERED_LCI_FROM_META.stat().st_mtime:
        raise RuntimeError(
            f"Stale DFE table detected: {DEFAULT_LCI} is older than {FILTERED_LCI_FROM_META}. "
            "Run notebook 3 to regenerate the DFE table for the current filtered selection before XML generation."
        )
else:
    DEFAULT_LCI = FILTERED_LCI_FROM_META
XML_EXPORTS_ROOT = PROJECT_DIR / "outputs/05_xml_exports_crop_lci"
SUMMARY_FOLDER_NAME = f"summary_{SUMMARY_TOKEN}"
DEFAULT_OUT_DIR = XML_EXPORTS_ROOT / SUMMARY_FOLDER_NAME
DEFAULT_UNCERTAINTY = DEFAULT_LCI.with_name(f"{DEFAULT_LCI.stem}_uncertainty.csv")

TEMPLATE_XML_PATH = DEFAULT_TEMPLATE
LCI_TABLE_PATH = DEFAULT_LCI
UNCERTAINTY_TABLE_PATH = DEFAULT_UNCERTAINTY
OUTPUT_DIR = DEFAULT_OUT_DIR

DFE_FACTORS_PATH = PROJECT_DIR / "inputs/03-05_dfe_factors.yml"
IRRIGATION_SOURCE_OPTIONS = {"lake", "river", "well"}
DEFAULT_IRRIGATION_SOURCE = "river"
IRRIGATION_SOURCE_COLUMN_CANDIDATES = ("Irrigation_source", "irrigation_water_source", "Irrig_source")
IRRIGATION_SOURCE_VALUE_COLUMNS = {
    "lake": "Irrig_lake_m3",
    "river": "Irrig_river_m3",
    "well": "Irrig_well_m3",
}
SEED_EXCHANGE_BY_CROP_TYPE = {
    "cereal": "Seed, cereal",
    "vegetable": "Seed, vegetable",
    "oilseed": "Seed, oilseed",
}
DEFAULT_SEED_KGHA_BY_CROP_TYPE = {
    "cereal": 150.0,
    "fruit": 5.0,
    "vegetable": 3.0,
    "oilseed": 8.0,
    "grass": 30.0,
}
DEFAULT_SEED_CROP_TYPE = "cereal"
DEFAULT_FUEL_MJ_PER_L = 38.0
DEFAULT_XML_REVIEWER_NAME = "Angel AvadÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¾ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¾ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¾ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¾ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¦ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¦ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¾ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¾ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¾ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¦ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¦ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¾ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¦ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¦ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â‚¬Å¾Ã‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¦Ãƒâ€šÃ‚Â¡ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­"
DEFAULT_XML_VALIDATION_METADATA = {
    "reviewStatus": "Final",
    "reviewer": DEFAULT_XML_REVIEWER_NAME,
    "proofReadingValidator": {
        "name": DEFAULT_XML_REVIEWER_NAME,
        "email": "angel.avadi@irta.cat",
        "companyCode": "1234",
    },
}
DEFAULT_CROP_RESIDUE_WASTE_FACTORS = {
    "enabled": True,
    "use_row_straw_if_available": True,
    "default_crop_type": "cereal",
    "residue_to_yield_ratio_by_crop_type": {
        "cereal": 1.3,
        "vegetable": 1.0,
        "fruit": 0.6,
        "oilseed": 2.5,
        "grass": 1.0,
    },
    "recoverable_fraction_by_crop_type": {
        "cereal": 0.4,
        "vegetable": 0.6,
        "fruit": 0.3,
        "oilseed": 0.5,
        "grass": 0.4,
    },
}
LAND_NORMALIZED_FLOW_NAMES = {
    "occupation, annual crop",
    "occupation, annual crop, ec",
    "transformation, from annual crop",
    "transformation, from annual crop, ec",
    "transformation, to annual crop",
    "transformation, to annual crop, ec",
    "occupation, permanent crop",
    "occupation, permanent crop, ec",
    "transformation, from permanent crop",
    "transformation, from permanent crop, ec",
    "transformation, to permanent crop",
    "transformation, to permanent crop, ec",
}

def _land_use_name_mapping_for_row(row: pd.Series) -> dict[str, str]:
    is_permanent = _row_str(row, "Category", default="").strip().lower() == "permanent"
    target = {
        "occupation": "Occupation, permanent crop, EC" if is_permanent else "Occupation, annual crop, EC",
        "from": "Transformation, from permanent crop, EC" if is_permanent else "Transformation, from annual crop, EC",
        "to": "Transformation, to permanent crop, EC" if is_permanent else "Transformation, to annual crop, EC",
    }
    return {
        "occupation, annual crop": target["occupation"],
        "occupation, annual crop, ec": target["occupation"],
        "occupation, permanent crop": target["occupation"],
        "occupation, permanent crop, ec": target["occupation"],
        "transformation, from annual crop": target["from"],
        "transformation, from annual crop, ec": target["from"],
        "transformation, from permanent crop": target["from"],
        "transformation, from permanent crop, ec": target["from"],
        "transformation, to annual crop": target["to"],
        "transformation, to annual crop, ec": target["to"],
        "transformation, to permanent crop": target["to"],
        "transformation, to permanent crop, ec": target["to"],
    }

def load_irrigation_water_source(config_path: Path, default=DEFAULT_IRRIGATION_SOURCE):
    source = default
    if config_path.exists():
        try:
            with open(config_path, "r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f) or {}
            source = str(cfg.get("assumptions", {}).get("irrigation_water_source", default)).strip().lower()
        except Exception as exc:
            print(f"Warning: failed to read irrigation source from {config_path}: {exc}")
            source = default
    if source not in IRRIGATION_SOURCE_OPTIONS:
        print(f"Warning: invalid irrigation source '{source}' in {config_path}; using default '{default}'.")
        source = default
    return source

def load_dfe_assumptions(config_path: Path):
    assumptions = {}
    if config_path.exists():
        try:
            with open(config_path, "r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f) or {}
            assumptions = cfg.get("assumptions", {}) or {}
        except Exception as exc:
            print(f"Warning: failed to read assumptions from {config_path}: {exc}")
    return assumptions

def resolve_xml_validation_metadata(assumptions: dict):
    assumptions = assumptions or {}
    xml_meta = assumptions.get("xml_validation_metadata", {}) or {}
    proof_raw = xml_meta.get("proofReadingValidator", {}) or {}

    reviewer_name = str(
        xml_meta.get("reviewer")
        or xml_meta.get("reviewer_name")
        or assumptions.get("reviewer")
        or assumptions.get("reviewer_name")
        or DEFAULT_XML_REVIEWER_NAME
    ).strip() or DEFAULT_XML_REVIEWER_NAME

    proofreader_name = str(
        proof_raw.get("name")
        or xml_meta.get("proofreader")
        or xml_meta.get("proofreader_name")
        or assumptions.get("proofreader")
        or assumptions.get("proofreader_name")
        or reviewer_name
    ).strip() or reviewer_name

    proofreader_email = str(
        proof_raw.get("email")
        or xml_meta.get("proofreader_email")
        or assumptions.get("proofreader_email")
        or DEFAULT_XML_VALIDATION_METADATA["proofReadingValidator"]["email"]
    ).strip()
    proofreader_company = str(
        proof_raw.get("companyCode")
        or xml_meta.get("proofreader_company_code")
        or assumptions.get("proofreader_company_code")
        or DEFAULT_XML_VALIDATION_METADATA["proofReadingValidator"]["companyCode"]
    ).strip()

    review_status = str(
        xml_meta.get("reviewStatus")
        or assumptions.get("review_status")
        or DEFAULT_XML_VALIDATION_METADATA["reviewStatus"]
    ).strip() or DEFAULT_XML_VALIDATION_METADATA["reviewStatus"]

    return {
        "reviewStatus": review_status,
        "reviewer": reviewer_name,
        "proofReadingValidator": {
            "name": proofreader_name,
            "email": proofreader_email,
            "companyCode": proofreader_company,
        },
    }


DFE_ASSUMPTIONS = load_dfe_assumptions(DFE_FACTORS_PATH)
IRRIGATION_WATER_SOURCE = load_irrigation_water_source(DFE_FACTORS_PATH)
SEED_KGHA_BY_CROP_TYPE = dict(DEFAULT_SEED_KGHA_BY_CROP_TYPE)
SEED_KGHA_BY_CROP_TYPE.update(DFE_ASSUMPTIONS.get("seed_kgha_by_crop_type", {}) or {})
DEFAULT_SEED_CROP_TYPE = str(DFE_ASSUMPTIONS.get("default_seed_crop_type", DEFAULT_SEED_CROP_TYPE)).strip().lower() or DEFAULT_SEED_CROP_TYPE
FUEL_MJ_PER_L = float(DFE_ASSUMPTIONS.get("fuel_mj_per_l", DEFAULT_FUEL_MJ_PER_L) or DEFAULT_FUEL_MJ_PER_L)
CROP_RESIDUE_WASTE_FACTORS = dict(DEFAULT_CROP_RESIDUE_WASTE_FACTORS)
CROP_RESIDUE_WASTE_FACTORS.update(DFE_ASSUMPTIONS.get("crop_residue_waste", {}) or {})
XML_VALIDATION_METADATA = resolve_xml_validation_metadata(DFE_ASSUMPTIONS)

assert TEMPLATE_XML_PATH.exists(), f"Template XML not found: {TEMPLATE_XML_PATH}"
assert LCI_TABLE_PATH.exists(), (
    f"LCI table not found: {LCI_TABLE_PATH}. "
    f"Inherited strategy: {SUMMARY_STRATEGY} (source: {SUMMARY_SOURCE})."
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Inherited summary strategy: {SUMMARY_STRATEGY} (source: {SUMMARY_SOURCE})")
display(Markdown(f"### Inherited strategy: `{SUMMARY_STRATEGY}`"))
print(f"Metadata file: {LATEST_FILTERED_SUMMARY_META_PATH if LATEST_FILTERED_SUMMARY_META_PATH.exists() else '(missing)'}")
print(f"Template: {TEMPLATE_XML_PATH}")
print(f"LCI table: {LCI_TABLE_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
if UNCERTAINTY_TABLE_PATH.exists():
    print(f"Uncertainty table: {UNCERTAINTY_TABLE_PATH}")
    print("Uncertainty integration: enabled")
else:
    print(f"Uncertainty table: {UNCERTAINTY_TABLE_PATH}")
    print("Uncertainty integration: NOT used (table not found; falling back to point values where min=max)")
print(f"Irrigation water source: {IRRIGATION_WATER_SOURCE}")
print(f"Default seed crop type: {DEFAULT_SEED_CROP_TYPE}")
print(f"Fuel conversion factor (MJ/L): {FUEL_MJ_PER_L}")





from scripts.pipeline_manifest import new_run_id, make_snapshot_copy, build_manifest_record, append_manifest_record


In [ ]:
df = pd.read_csv(LCI_TABLE_PATH)

uncertainty_path = UNCERTAINTY_TABLE_PATH
if not uncertainty_path.exists():
    uncertainty_path = LCI_TABLE_PATH.with_name(f"{LCI_TABLE_PATH.stem}_uncertainty.csv")

if uncertainty_path.exists():
    print(f"Using uncertainty table: {uncertainty_path}")
    df_unc = pd.read_csv(uncertainty_path)
    unc_cols = [
        c for c in df_unc.columns
        if isinstance(c, str)
        and (
            (c.endswith("__minValue") and c.count("__minValue") == 1)
            or (c.endswith("__maxValue") and c.count("__maxValue") == 1)
        )
    ]
    if len(df_unc) == len(df) and unc_cols:
        # Preferred path: uncertainty exports are row-aligned with the LCI table.
        # Overwrite-or-add in place so pre-existing helper columns (for example
        # SOC_mean_rate_kgChayr__minValue on the DFE table) do not become
        # duplicate labels that later collapse uncertainty to point values.
        df = df.reset_index(drop=True).copy()
        df_unc_vals = df_unc[unc_cols].reset_index(drop=True)
        for c in unc_cols:
            df[c] = df_unc_vals[c].to_numpy()
    else:
        merge_keys = [c for c in ['Region', 'Province', 'Crop', 'Category'] if c in df.columns and c in df_unc.columns]
        if merge_keys and unc_cols:
            # Avoid row explosion on non-unique keys in uncertainty table.
            right = df_unc[merge_keys + unc_cols].drop_duplicates(subset=merge_keys, keep='first')
            base = df.reset_index().rename(columns={'index': '__row_id'})
            df = base.merge(right, on=merge_keys, how='left').sort_values('__row_id').drop(columns='__row_id').reset_index(drop=True)
        elif unc_cols:
            for c in unc_cols:
                if c not in df.columns:
                    df[c] = df_unc[c]

if 'Yield_kgha' in df.columns:
    _yield_num = pd.to_numeric(df['Yield_kgha'], errors='coerce')
    _before_rows = len(df)
    df = df.loc[_yield_num > 0].copy()
    _removed_rows = _before_rows - len(df)
    if _removed_rows:
        print(f"Filtered out {_removed_rows} rows with non-positive/invalid Yield_kgha from CSV table.")
else:
    print("Warning: Yield_kgha column not found; no yield-based row filtering applied.")

if uncertainty_path.exists():
    _unc_min_cols = [c for c in df.columns if c.endswith("__minValue") and c.count("__minValue") == 1]
    _unc_rows = int(df[_unc_min_cols].notna().any(axis=1).sum()) if _unc_min_cols else 0
    _unc_no_rows = len(df) - _unc_rows
    print(f"Uncertainty columns: {len(_unc_min_cols)}")
    print(f"Rows with uncertainty coverage: {_unc_rows}/{len(df)}")
    if _unc_no_rows:
        print(f"Warning: {_unc_no_rows} rows have no uncertainty bounds")
print(f"Rows: {len(df):,}  Columns: {len(df.columns)}")
display(df.head(3))


In [ ]:
def normalize_key(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def to_numeric_or_default(v, default=0.0):
    try:
        if pd.isna(v):
            return float(default)
        return float(v)
    except Exception:
        return float(default)


PLACEHOLDER_NAME_PARTS = {
    "all_crops_in_group",
    "all_regions_confounded",
    "all_provinces_confounded",
}


def _normalize_name_token(part) -> str:
    txt = str(part).strip().lower()
    txt = re.sub(r"[^a-z0-9]+", "_", txt).strip("_")
    return txt


def _clean_name_part(part) -> str:
    txt = str(part).strip()
    if not txt or txt == "nan":
        return ""
    if _normalize_name_token(txt) in PLACEHOLDER_NAME_PARTS:
        return ""
    return txt


def safe_name(*parts):
    txt = "_".join([part for p in parts if (part := _clean_name_part(p))])
    txt = re.sub(r"[^A-Za-z0-9._-]+", "_", txt).strip("_")
    return txt[:180] if txt else "record"


def _normalized_strategy_value(raw: str, token: str) -> str:
    raw = str(raw or '').strip()
    raw_lc = raw.lower()
    if token == "cropping_system":
        if "mono" in raw_lc:
            return "mono"
        if "asso" in raw_lc:
            return "associated"
    if token == "irrig_m3_class":
        if "= 0" in raw_lc or "=0" in raw_lc:
            return "irrig_0"
        if "<> 0" in raw_lc or "!= 0" in raw_lc or "> 0" in raw_lc:
            return "irrig_gt0"
    return raw


def _clean_display_part(value) -> str:
    raw = str(value or '').strip()
    if not raw or raw.lower() == 'nan':
        return ''
    raw_lc = raw.lower()
    if raw_lc in {'(unknown)', 'unknown'}:
        return ''
    return raw


def _is_confounded_placeholder(value) -> bool:
    raw_lc = str(value or '').strip().lower()
    return (
        raw_lc.startswith('(all ') or
        raw_lc.endswith(' confounded)') or
        raw_lc in {'(all regions confounded)', '(all provinces confounded)', '(all crops in group)'}
    )


def _aggregation_display(summary_token: str) -> str:
    token = str(summary_token or '').strip().lower()
    return token or "unknown"


def _strategy_context_for_row(row: pd.Series, summary_token: str):
    token = str(summary_token or '').strip().lower()
    strategy_to_col = {
        'cropping_system': ('cropping system', 'system', 'Cropping_system'),
        'irrig_m3_class': ('irrigation class', 'irrig', 'Irrig_m3_class'),
        'farm_size_class': ('farm size class', 'farm', 'Farm_size_class'),
        'crop_group': ('crop group', 'group', 'Crop_group'),
        'crop_group_national': ('crop group', 'group', 'Crop_group'),
    }
    cfg = strategy_to_col.get(token)
    if not cfg:
        return None
    label, slug_label, col = cfg
    if col not in row.index:
        return None
    raw = _clean_display_part(row.get(col, ''))
    if not raw:
        return None
    value = _normalized_strategy_value(raw, token)
    if not value:
        return None
    return {'label': label, 'slug_label': slug_label, 'value': value}


def _summary_count_key(raw_value):
    try:
        return int(round(float(raw_value)))
    except Exception:
        return None


_ORIGINAL_SUMMARY_CATEGORY_BY_COUNT_AND_GROUP = {}
_SUMMARY_GROUP_DUPLICATION_COUNT = {}
if str(SUMMARY_TOKEN).strip().lower() == 'crop_group_national':
    _source_filtered_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{SUMMARY_TOKEN}.csv'
    if _source_filtered_csv.exists():
        try:
            _source_df = pd.read_csv(_source_filtered_csv)
            if {'count', 'Crop_group', 'Category'}.issubset(_source_df.columns):
                for _rec in _source_df[['count', 'Crop_group', 'Category']].to_dict('records'):
                    _group = _clean_display_part(_rec.get('Crop_group', ''))
                    _category = _clean_display_part(_rec.get('Category', ''))
                    _count_key = _summary_count_key(_rec.get('count'))
                    if _group and _category and _count_key is not None:
                        _ORIGINAL_SUMMARY_CATEGORY_BY_COUNT_AND_GROUP[(_count_key, _group.lower())] = _category
                        _SUMMARY_GROUP_DUPLICATION_COUNT[_group.lower()] = _SUMMARY_GROUP_DUPLICATION_COUNT.get(_group.lower(), 0) + 1
        except Exception as exc:
            print(f'Warning: failed loading summary-category lookup from {_source_filtered_csv}: {exc}')


def _strategy8_group_and_system_for_row(row: pd.Series) -> tuple[str, str]:
    group = _clean_display_part(row.get('Crop_group', ''))
    category = _clean_display_part(row.get('Category', ''))
    group_lc = group.lower()
    if group and (_normalize_name_token(category) == _normalize_name_token(group) or not category):
        key = (_summary_count_key(row.get('count')), group_lc)
        category = _ORIGINAL_SUMMARY_CATEGORY_BY_COUNT_AND_GROUP.get(key, category)
    return group, category


def _display_name_and_slug_parts_for_row(row: pd.Series, summary_token: str, row_number=None):
    display_parts = []
    slug_parts = [f'{int(row_number):05d}'] if row_number is not None else []

    token = str(summary_token or '').strip().lower()
    if token == 'crop_group_national':
        group, category = _strategy8_group_and_system_for_row(row)
        if group:
            display_parts.append(group)
            slug_parts.append(f"group_{safe_name(group).lower()}")

        display_parts.append(f'aggregation: {_aggregation_display(summary_token)}')
        return display_parts, slug_parts

    ctx = _strategy_context_for_row(row, summary_token)
    if ctx:
        display_parts.append(f"{ctx['label']}: {ctx['value']}")
        slug_parts.append(f"{ctx['slug_label']}_{safe_name(ctx['value']).lower()}")

    crop = _clean_display_part(row.get('Crop', ''))
    if crop and not _is_confounded_placeholder(crop):
        display_parts.append(f'crop: {crop}')
        slug_parts.append(f"crop_{safe_name(crop).lower()}")

    category = _clean_display_part(row.get('Category', ''))
    ctx_value_lc = str((ctx or {}).get('value', '')).strip().lower()
    if category and category.lower() != ctx_value_lc:
        display_parts.append(f'category: {category}')
        slug_parts.append(f"category_{safe_name(category).lower()}")

    region = _clean_display_part(row.get('Region', ''))
    if region and not _is_confounded_placeholder(region):
        display_parts.append(f'region: {region}')
        slug_parts.append(f"region_{safe_name(region).lower()}")

    province = _clean_display_part(row.get('Province', ''))
    if province and not _is_confounded_placeholder(province):
        display_parts.append(f'province: {province}')
        slug_parts.append(f"province_{safe_name(province).lower()}")

    display_parts.append(f'aggregation: {_aggregation_display(summary_token)}')
    return display_parts, slug_parts


def _process_name_for_row(row: pd.Series, summary_token: str) -> str:
    display_parts, _ = _display_name_and_slug_parts_for_row(row, summary_token)
    return ' | '.join(display_parts)[:255] if display_parts else 'record'


def _filename_for_row(row: pd.Series, summary_token: str, row_number=None) -> str:
    _, slug_parts = _display_name_and_slug_parts_for_row(row, summary_token, row_number=row_number)
    return safe_name(*slug_parts) + '.xml'


def get_root_and_ns(template_xml_path: Path):
    tree = ET.parse(template_xml_path)
    root = tree.getroot()
    m = re.match(r"\{(.+)\}", root.tag)
    ns = {"spold": m.group(1)} if m else {}
    return tree, root, ns


def get_exchange_elements(root, ns):
    if ns:
        return root.findall(".//spold:exchange", ns)
    return root.findall(".//exchange")


def build_comment_mapping(template_xml_path: Path, df_columns, manual_overrides=None):
    manual_overrides = manual_overrides or {}
    _, root, ns = get_root_and_ns(template_xml_path)
    exchanges = get_exchange_elements(root, ns)

    col_lookup = {normalize_key(c): c for c in df_columns}
    mapped = {}
    unmatched = []

    for ex in exchanges:
        comment = ex.attrib.get("generalComment", "").strip()
        if not comment:
            continue

        if comment in manual_overrides:
            mapped[comment] = manual_overrides[comment]
            continue

        nk = normalize_key(comment)
        if nk in col_lookup:
            mapped[comment] = col_lookup[nk]
        else:
            unmatched.append(comment)

    mapping_df = pd.DataFrame({
        "xml_generalComment": list(mapped.keys()),
        "table_column": list(mapped.values()),
    }).sort_values(["xml_generalComment"]).reset_index(drop=True)

    unmatched = sorted(set(unmatched))
    return mapped, mapping_df, unmatched


def _row_has_col(row: pd.Series, col: str) -> bool:
    return col in row.index and not pd.isna(row.get(col))


def _row_str(row: pd.Series, *candidates, default="") -> str:
    for c in candidates:
        if c in row.index and not pd.isna(row.get(c)):
            return str(row.get(c)).strip()
    return default


def _row_float(row: pd.Series, *candidates, default=0.0):
    for c in candidates:
        if c in row.index:
            v = row.get(c)
            if not pd.isna(v):
                try:
                    return float(v)
                except Exception:
                    pass
    return float(default)


def _collect_indexed_pairs(row: pd.Series, name_prefix: str, value_prefix: str, max_items=20):
    pairs = []
    for i in range(1, max_items + 1):
        k_name = f"{name_prefix}_{i}"
        k_val = f"{value_prefix}_{i}"
        if k_name not in row.index and k_val not in row.index:
            continue
        name = _row_str(row, k_name, default="").strip()
        if not name or name.lower() in {"missing", "nan"}:
            continue
        pairs.append((name, _row_float(row, k_val, default=0.0)))
    return pairs


def collect_pesticides(row: pd.Series, max_items=20):
    return [(name.lower(), dose) for name, dose in _collect_indexed_pairs(row, "Molecule_name", "Dose_kgha", max_items=max_items)]


def _norm_token(txt: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(txt).lower())


def _load_yaml_safely(path: Path):
    if not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return yaml.safe_load(f) or {}
    except Exception as exc:
        print(f"Warning: failed reading YAML {path}: {exc}")
        return {}


ESPAC_COEFFICIENTS_PATH = PROJECT_DIR / "inputs/02-05_espac_lci_coefficients.yml"
ESPAC_COEFFS = _load_yaml_safely(ESPAC_COEFFICIENTS_PATH)
DFE_FACTORS = _load_yaml_safely(DFE_FACTORS_PATH)

# Crop-level molecule shares from YAML (single source of truth)
PESTICIDE_SHARES_BY_CROP = ESPAC_COEFFS.get("pesticide_molecule_shares_by_crop", {}) or {}
PESTICIDE_SHARES_BY_CROP_GROUP = ESPAC_COEFFS.get("pesticide_molecule_shares_by_crop_group", {}) or {}

# Prefix mapping for crop-type-specific partition factors in inputs/03-05_dfe_factors.yml
# Default prefix is cereals ("c") when no specific type mapping exists.
CROP_TYPE_TO_PARTITION_PREFIX = {
    "cereal": "c",
    "fruit": "f",
    "vegetable": "v",
}
DEFAULT_PARTITION_PREFIX = "c"

# XML partition comments to partition suffix names used in inputs/03-05_dfe_factors.yml
PARTITION_COMMENT_TO_SUFFIX = {
    "c_PrimAir": "PrimAir",
    "c_PrimWater": "PrimWater",
    "soil_factor": "PrimSoil",
    "c_PrimSoil": "PrimSoil",
    "c_PrimCrop": "PrimCrop",
    "c_PrimOffField": "PrimOffField",
}


def _partition_factor_for_row(row: pd.Series, partition_comment: str, default=0.0) -> float:
    suffix = PARTITION_COMMENT_TO_SUFFIX.get(str(partition_comment).strip())
    if not suffix:
        return float(default)
    crop_type = _infer_crop_type_for_row(row)
    prefix = CROP_TYPE_TO_PARTITION_PREFIX.get(crop_type, DEFAULT_PARTITION_PREFIX)
    knime = DFE_FACTORS.get("knime_factors", {}) or {}
    key = f"{prefix}_{suffix}"
    # Fallback to cereal factors if crop type partition is missing.
    raw = knime.get(key, knime.get(f"{DEFAULT_PARTITION_PREFIX}_{suffix}", default))
    return float(raw if raw is not None else default)


def _row_total_pesticides_kgha(row: pd.Series) -> float:
    # Prefer explicit total, fallback to class-wise sum.
    if _row_has_col(row, "Total_pesticides_kgha"):
        return _row_float(row, "Total_pesticides_kgha", default=0.0)
    return (
        _row_float(row, "Insecticide_kgha", default=0.0)
        + _row_float(row, "Herbicide_kgha", default=0.0)
        + _row_float(row, "Fungicide_kgha", default=0.0)
    )


def _pesticide_doses_by_molecule_for_row(row: pd.Series):
    # 1) Preferred: molecule shares by crop from inputs/02-05_espac_lci_coefficients.yml
    crop_name = _row_str(row, "Crop", default="").upper()
    crop_payload = PESTICIDE_SHARES_BY_CROP.get(crop_name, {}) or {}
    molecules = crop_payload.get("molecules", []) or []
    # 1b) Fallback: group-level averages when Crop is a placeholder (e.g. crop_group_national strategy)
    if not molecules:
        crop_group = canonical_crop_group_token(_row_str(row, "Crop_group", default=""))
        crop_payload = PESTICIDE_SHARES_BY_CROP_GROUP.get(crop_group, {}) or {}
        molecules = crop_payload.get("molecules", []) or []
    total_pest = _row_total_pesticides_kgha(row)
    if molecules and total_pest > 0:
        out = {}
        for item in molecules:
            mol = str(item.get("molecule", "")).strip()
            share = float(item.get("contribution_percent", 0.0) or 0.0)
            if not mol or share <= 0:
                continue
            out[_norm_token(mol)] = out.get(_norm_token(mol), 0.0) + total_pest * (share / 100.0)
        if out:
            return out

    # 2) Backward-compatibility fallback to per-row indexed molecule columns.
    legacy = collect_pesticides(row)
    if legacy:
        out = {}
        for mol, dose in legacy:
            k = _norm_token(mol)
            out[k] = out.get(k, 0.0) + float(dose)
        return out

    return {}


def collect_transport_totals(row: pd.Series, max_items=20):
    totals = {}
    for name, tkm in _collect_indexed_pairs(row, "Transport_type", "Transport_tkm", max_items=max_items):
        key = str(name).strip()
        totals[key] = totals.get(key, 0.0) + float(tkm)
    return totals


def _get_irrigation_exchange_source(exchange_name: str) -> str:
    ex_name = str(exchange_name).strip().lower()
    if not ex_name.startswith("water,"):
        return ""
    source = ex_name.split(",", 1)[1].split(",", 1)[0].strip()
    return source if source in IRRIGATION_SOURCE_OPTIONS else ""


def _resolve_irrigation_source_for_row(row: pd.Series, default_source: str) -> str:
    for col in IRRIGATION_SOURCE_COLUMN_CANDIDATES:
        if col in row.index and not pd.isna(row.get(col)):
            val = str(row.get(col)).strip().lower()
            if val in IRRIGATION_SOURCE_OPTIONS:
                return val
    return default_source


def _infer_crop_type_for_row(row: pd.Series) -> str:
    explicit = _row_str(row, "Custom_classification_3", default="").lower()
    if explicit in {"cereal", "fruit", "vegetable", "oilseed", "grass"}:
        return explicit

    explicit_dfe = _row_str(row, "crop_type_dfe", default="").lower()
    if explicit_dfe in {"cereal", "fruit", "vegetable", "oilseed", "grass"}:
        return explicit_dfe

    crop_name = _row_str(row, "Crop", default="").upper()
    keyword_groups = (DFE_ASSUMPTIONS.get("crop_type_inference", {}) or {}).get("keyword_groups", {}) or {}
    for crop_type, keywords in keyword_groups.items():
        for kw in (keywords or []):
            if str(kw).upper() in crop_name:
                return str(crop_type).lower()

    category = _row_str(row, "Category", default="").lower()
    fallback_by_category = (DFE_ASSUMPTIONS.get("crop_type_inference", {}) or {}).get("fallback_by_category", {}) or {}
    if category in fallback_by_category:
        return str(fallback_by_category[category]).lower()

    return DEFAULT_SEED_CROP_TYPE if DEFAULT_SEED_CROP_TYPE in SEED_KGHA_BY_CROP_TYPE else "cereal"


def _resolve_seed_value_and_exchange_for_row(row: pd.Series):
    crop_type = _infer_crop_type_for_row(row)
    default_seed = float(SEED_KGHA_BY_CROP_TYPE.get(crop_type, SEED_KGHA_BY_CROP_TYPE.get(DEFAULT_SEED_CROP_TYPE, 0.0)))
    seed_from_row = None
    if "Seed_kgha" in row.index and not pd.isna(row.get("Seed_kgha")):
        try:
            seed_from_row = float(row.get("Seed_kgha"))
        except Exception:
            seed_from_row = None
    seed_value = seed_from_row if (seed_from_row is not None and seed_from_row > 0.0) else default_seed
    target_exchange = SEED_EXCHANGE_BY_CROP_TYPE.get(crop_type, SEED_EXCHANGE_BY_CROP_TYPE.get(DEFAULT_SEED_CROP_TYPE, "Seed, cereal"))
    return crop_type, seed_value, target_exchange


def _compute_crop_residue_waste_kgha(row: pd.Series, crop_type: str):
    factors = CROP_RESIDUE_WASTE_FACTORS
    if not bool(factors.get("enabled", True)):
        return 0.0

    if bool(factors.get("use_row_straw_if_available", True)) and _row_has_col(row, "Straw_kgha"):
        v = _row_float(row, "Straw_kgha", default=0.0)
        if v > 0.0:
            return float(v)

    ctype = crop_type or str(factors.get("default_crop_type", "cereal")).lower()
    ratio_map = factors.get("residue_to_yield_ratio_by_crop_type", {}) or {}
    rec_map = factors.get("recoverable_fraction_by_crop_type", {}) or {}
    ratio = float(ratio_map.get(ctype, ratio_map.get(str(factors.get("default_crop_type", "cereal")).lower(), 0.0)) or 0.0)
    recoverable = float(rec_map.get(ctype, rec_map.get(str(factors.get("default_crop_type", "cereal")).lower(), 1.0)) or 0.0)
    yield_kgha = _row_float(row, "Yield_kgha", default=0.0)
    return float(max(0.0, yield_kgha * ratio * recoverable))


YIELD_AREA_COMMENT = "Model_annual_crops, yield per 1 ha, actual area (ha): Area_ha"
SPECIAL_EXTRA_INPUT_COMMENTS = {"CH4", "PO4", "SOC_mean_rate_kgChayr", "Straw_kgha", "Ch"}


DIRECT_FIELD_EMISSION_COMMENT_ALIASES = {
    "NH3": "kgNH3N_ha",
    "N2O": "kgN2ON_ha",
    "NOx": "kgNOxN_ha",
    "CO2": "kgCO2_fert_ha",
    "NO3": "kgNO3N_ha",
    "P": "kgP_total",
    "Cd": "Cd_kg_ha",
    "Cu": "Cu_kg_ha",
    "Zn": "Zn_kg_ha",
    "Pb": "Pb_kg_ha",
    "Ni": "Ni_kg_ha",
    "Cr": "Cr_kg_ha",
    "Hg": "Hg_kg_ha",
}


LEGACY_RULE_COMMENTS = {
    "Fuel", "Truck", "Tractor",
    "NH3", "N2O", "NOx", "CO2", "NO3", "P",
    "Cd", "Cu", "Zn", "Pb", "Ni", "Cr", "Hg",
    "PP", "PE", "PET", "LDPE", "Kraft", "Cardboard",
    "Cattle, solid", "Cattle, liquid", "Swine, liquid", "Poultry, droppings",
    "Seed, cereal", "Seed, vegetable", "Seed, oilseed",
    "Compost", "Digestate", "Compost_digestate",
    "c_PrimAir", "c_PrimWater", "soil_factor",
    YIELD_AREA_COMMENT,
}


def classify_unmatched_comments(unmatched_comments, df_columns):
    df_cols = set(df_columns)
    legacy_handled = []
    needs_extra_inputs = []
    truly_unmatched = []

    has_molecule_cols = any(str(c).startswith("Molecule_name_") for c in df_cols)
    has_pest_table_cols = any(
        c in df_cols for c in {"Total_pesticides_kgha", "Insecticide_kgha", "Herbicide_kgha", "Fungicide_kgha"}
    )
    has_yaml_shares = bool(PESTICIDE_SHARES_BY_CROP)
    has_transport_cols = any(str(c).startswith("Transport_type_") for c in df_cols)

    for comment in unmatched_comments:
        if comment in {"c_PrimAir", "c_PrimWater", "soil_factor"}:
            if (has_yaml_shares and has_pest_table_cols) or has_molecule_cols:
                legacy_handled.append(comment)
            else:
                needs_extra_inputs.append(comment)
            continue
        if comment in {"Truck", "Tractor"}:
            if has_transport_cols:
                legacy_handled.append(comment)
            else:
                needs_extra_inputs.append(comment)
            continue
        if comment == YIELD_AREA_COMMENT:
            legacy_handled.append(comment)
            continue
        if comment in SPECIAL_EXTRA_INPUT_COMMENTS:
            if comment == "Ch" and ("Cr" in df_cols or "Ch" in df_cols):
                legacy_handled.append(comment)
            elif comment in df_cols:
                legacy_handled.append(comment)
            else:
                needs_extra_inputs.append(comment)
            continue
        if comment in LEGACY_RULE_COMMENTS:
            legacy_handled.append(comment)
        else:
            truly_unmatched.append(comment)

    return {
        "legacy_handled": sorted(legacy_handled),
        "needs_extra_inputs": sorted(needs_extra_inputs),
        "truly_unmatched": sorted(truly_unmatched),
    }


def apply_legacy_rule_to_exchange(exchange, row: pd.Series):
    general_comment = str(exchange.attrib.get("generalComment", "")).strip()
    ex_name = str(exchange.attrib.get("name", "")).strip().lower()
    if not general_comment:
        return None

    seed_crop_type, seed_kgha, target_seed_exchange = _resolve_seed_value_and_exchange_for_row(row)

    # Yield/area placeholder in template (legacy KNIME replaced XML string placeholders)
    if general_comment == YIELD_AREA_COMMENT:
        mean_yield = _row_float(row, "Yield_kgha", default=0.0)
        lo_yield, hi_yield = _resolve_uncertainty_bounds(row, "Yield_kgha", fallback_value=mean_yield)
        if _row_has_col(row, "Area_ha"):
            area_txt = f"{_row_float(row, 'Area_ha', default=0.0):.2f}"
            base_comment = general_comment.replace("Area_ha", area_txt)
            if str(exchange.attrib.get("number", "")).strip() == "1":
                exchange.attrib["generalComment"] = (
                    f"{base_comment}, yield range: {float(lo_yield):.2f} - {float(hi_yield):.2f}"
                )
            else:
                exchange.attrib["generalComment"] = base_comment
        return mean_yield

    # Comments that require extra precomputed columns when available
    if general_comment in {"CH4", "PO4", "SOC_mean_rate_kgChayr"}:
        if _row_has_col(row, general_comment):
            return _row_float(row, general_comment, default=0.0)
    if general_comment == "Straw_kgha":
        return _compute_crop_residue_waste_kgha(row, seed_crop_type)

    # Legacy template typo alias: "Ch" comment actually refers to chromium (Cr)
    if general_comment == "Ch":
        if _row_has_col(row, "Ch"):
            return _row_float(row, "Ch", default=0.0)
        if _row_has_col(row, "Cr"):
            return _row_float(row, "Cr", default=0.0)

    # Packaging logic (legacy KNIME style)
    p1_type = _row_str(row, "Packaging_type1", default="")
    p1_mass = _row_float(row, "Packaging_type1_kgha", default=0.0)
    p2_type = _row_str(row, "Packaging_type2", default="")
    p2_mass = _row_float(row, "Packaging_type2_kgha", default=0.0)
    packaging_map = {
        "PP": (1.0 if p1_type == "PP" else 0.0) * p1_mass,
        "PE": (1.0 if p1_type == "PE" else 0.0) * p1_mass,
        "PET": (1.0 if p1_type == "PET" else 0.0) * p1_mass,
        "LDPE": (1.0 if p1_type == "LDPE" else 0.0) * p1_mass,
        "Kraft": (1.0 if p2_type == "Kraft" else 0.0) * p2_mass,
        "Cardboard": (1.0 if p2_type == "Cardboard" else 0.0) * p2_mass,
    }
    for key, value in packaging_map.items():
        if key in general_comment:
            return float(value)

    # Fuel logic: populate diesel burned exchange in MJ from Fuel_ha (supports MJ/ha, L, and L/ha units)
    fuel_type = _row_str(row, "Fuel_type", default="").lower()
    fuel_unit = _row_str(row, "Fuel_unit", default="").lower()
    fuel_ha = _row_float(row, "Fuel_ha", default=0.0)
    is_diesel_exchange = (general_comment == "Fuel") and ("diesel" in ex_name)
    is_liter_based = ("l" in fuel_unit)
    is_mj_based = ("mj" in fuel_unit)
    if is_diesel_exchange:
        if fuel_ha <= 0.0:
            return 0.0
        if is_mj_based:
            return float(fuel_ha)
        if is_liter_based or (not fuel_unit and ("diesel" in fuel_type or not fuel_type)):
            return float(fuel_ha * FUEL_MJ_PER_L)
        return float(fuel_ha * FUEL_MJ_PER_L)

    # Organic inputs aliases (column names differ from legacy script in this CSV)
    if general_comment == "Compost":
        return _row_float(row, "Compost_kgha", "Compost_kg_ha", default=0.0)
    if general_comment == "Digestate":
        return _row_float(row, "Digestate_kgha", "Digestate_kg_ha", default=0.0)
    if general_comment == "Compost_digestate":
        return _row_float(row, "Compost_kgha", "Compost_kg_ha", default=0.0) + _row_float(row, "Digestate_kgha", "Digestate_kg_ha", default=0.0)

    # Manure family placeholders
    manure_type = _row_str(row, "Manure_type", default="")
    manure_total = _row_float(row, "Manure_kgha", default=0.0)
    if general_comment == "Cattle, solid" and "Cattle, solid" in manure_type:
        return manure_total
    if general_comment == "Cattle, liquid" and "Cattle, liquid" in manure_type:
        return manure_total
    if general_comment == "Swine, liquid" and "Swine, liquid" in manure_type:
        return manure_total
    if general_comment == "Poultry, droppings" and "Poultry" in manure_type:
        return manure_total

    # Seed placeholders by inferred crop type; unknown types default to cereal exchange
    if general_comment in {"Seed, cereal", "Seed, vegetable", "Seed, oilseed"}:
        if general_comment == target_seed_exchange:
            return seed_kgha
        return 0.0

    # Direct emissions / trace elements if precomputed columns are available
    # First, map legacy XML comments (NH3, Cd, etc.) to DFE table columns (kgNH3N_ha, Cd_kg_ha, ...).
    if general_comment in DIRECT_FIELD_EMISSION_COMMENT_ALIASES:
        alias_col = DIRECT_FIELD_EMISSION_COMMENT_ALIASES[general_comment]
        if _row_has_col(row, alias_col):
            return _row_float(row, alias_col, default=0.0)
    if _row_has_col(row, general_comment):
        return _row_float(row, general_comment, default=0.0)

    # Pesticide molecule-specific emissions.
    # Coefficients are resolved by crop type from inputs/03-05_dfe_factors.yml (c/f/v; default cereals "c"),
    # and doses come from crop molecule shares in inputs/02-05_espac_lci_coefficients.yml.
    if general_comment in {"c_PrimAir", "soil_factor", "c_PrimWater"}:
        doses_by_molecule = _pesticide_doses_by_molecule_for_row(row)
        if doses_by_molecule:
            coeff = _partition_factor_for_row(row, general_comment, default=0.0)
            dose = doses_by_molecule.get(_norm_token(ex_name), 0.0)
            return float(dose * coeff / 100.0)

    # Transport tkm by type
    if general_comment in {"Truck", "Tractor"}:
        totals = collect_transport_totals(row)
        if general_comment in totals:
            return float(totals[general_comment])

    return None


def _to_uncertainty_float(raw) -> float | None:
    if isinstance(raw, (bool, np.bool_)):
        return None
    if raw is None:
        return None
    if isinstance(raw, str):
        token = raw.strip().lower()
        if token in {"", "true", "false", "nan", "inf", "-inf"}:
            return None
    if isinstance(raw, float) and (pd.isna(raw) or np.isinf(raw)):
        return None
    val = pd.to_numeric(pd.Series([raw]), errors="coerce").iloc[0]
    if pd.isna(val) or np.isinf(float(val)):
        return None
    return float(val)


def _resolve_uncertainty_columns(row: pd.Series, col: str) -> tuple[str | None, str | None]:
    base = str(col).strip()
    direct_min = f"{base}__minValue"
    direct_max = f"{base}__maxValue"
    if direct_min in row.index and direct_max in row.index:
        return direct_min, direct_max

    target_min = normalize_key(direct_min)
    target_max = normalize_key(direct_max)
    for name in row.index:
        if not isinstance(name, str):
            continue
        norm_name = normalize_key(name)
        if norm_name == target_min:
            cmin = name
        elif norm_name == target_max:
            cmax = name

    try:
        return cmin, cmax
    except NameError:
        return None, None


def _resolve_uncertainty_bounds(row: pd.Series, col: str, fallback_value: float) -> tuple[float, float]:
    cmin, cmax = _resolve_uncertainty_columns(row, col)
    lo = _to_uncertainty_float(row.get(cmin, np.nan)) if cmin else None
    hi = _to_uncertainty_float(row.get(cmax, np.nan)) if cmax else None
    if lo is None:
        lo = fallback_value
    if hi is None:
        hi = fallback_value
    lo2 = min(float(lo), float(hi))
    hi2 = max(float(lo), float(hi))
    return lo2, hi2


def _legacy_uncertainty_alias_col(general_comment: str) -> str | None:
    if general_comment in DIRECT_FIELD_EMISSION_COMMENT_ALIASES:
        return DIRECT_FIELD_EMISSION_COMMENT_ALIASES[general_comment]
    alias_map = {
        YIELD_AREA_COMMENT: 'Yield_kgha',
        'Compost': 'Compost_kg_ha',
        'Digestate': 'Digestate_kg_ha',
        'Compost_digestate': 'Compost_kg_ha',
        'Straw_kgha': 'Straw_kgha',
    }
    return alias_map.get(general_comment)


def _resolve_fuel_uncertainty_bounds(row: pd.Series, exchange_name: str, fallback_value: float) -> tuple[float, float]:
    fuel_unit = _row_str(row, "Fuel_unit", default="").lower()

    cmin = "Fuel_ha__minValue"
    cmax = "Fuel_ha__maxValue"
    lo_fuel_ha = _to_uncertainty_float(row.get(cmin))
    hi_fuel_ha = _to_uncertainty_float(row.get(cmax))

    if lo_fuel_ha is None or hi_fuel_ha is None:
        mean_fuel_ha = _row_float(row, "Fuel_ha", default=0.0)
        lo_fuel_ha, hi_fuel_ha = mean_fuel_ha, mean_fuel_ha

    ex_name = str(exchange_name).strip().lower()
    is_diesel_exchange = "diesel" in ex_name
    is_mj_based = "mj" in fuel_unit

    if (not is_diesel_exchange) or max(lo_fuel_ha, hi_fuel_ha) <= 0.0:
        return 0.0, 0.0

    if is_mj_based:
        lo = lo_fuel_ha
        hi = hi_fuel_ha
    else:
        lo = lo_fuel_ha * FUEL_MJ_PER_L
        hi = hi_fuel_ha * FUEL_MJ_PER_L

    return min(float(lo), float(hi)), max(float(lo), float(hi))



def _resolve_pesticide_emission_uncertainty_bounds(
    row: pd.Series,
    exchange_name: str,
    partition_comment: str,
    fallback_value: float,
) -> tuple[float, float]:
    crop_name = _row_str(row, "Crop", default="").upper()
    crop_payload = PESTICIDE_SHARES_BY_CROP.get(crop_name, {}) or {}
    molecules = crop_payload.get("molecules", []) or []
    # 1b) Fallback: group-level averages when Crop is a placeholder (e.g. crop_group_national strategy)
    if not molecules:
        crop_group = canonical_crop_group_token(_row_str(row, "Crop_group", default=""))
        crop_payload = PESTICIDE_SHARES_BY_CROP_GROUP.get(crop_group, {}) or {}
        molecules = crop_payload.get("molecules", []) or []

    if not molecules:
        return float(fallback_value), float(fallback_value)

    token = _norm_token(exchange_name)
    share_pct = 0.0
    for item in molecules:
        mol = str(item.get("molecule", "")).strip()
        if not mol:
            continue
        if _norm_token(mol) == token:
            share_pct += float(item.get("contribution_percent", 0.0) or 0.0)

    if share_pct <= 0.0:
        return float(fallback_value), float(fallback_value)

    cmin = "Total_pesticides_kgha__minValue"
    cmax = "Total_pesticides_kgha__maxValue"
    lo_tot = _to_uncertainty_float(row.get(cmin, np.nan))
    hi_tot = _to_uncertainty_float(row.get(cmax, np.nan))

    if lo_tot is None or hi_tot is None:
        total_pest = _row_total_pesticides_kgha(row)
        lo_tot, hi_tot = total_pest, total_pest





    coeff = _partition_factor_for_row(row, partition_comment, default=0.0)
    lo = lo_tot * (share_pct / 100.0) * coeff / 100.0
    hi = hi_tot * (share_pct / 100.0) * coeff / 100.0
    return min(float(lo), float(hi)), max(float(lo), float(hi))



def _resolve_straw_uncertainty_bounds(row: pd.Series, fallback_value: float) -> tuple[float, float]:
    # Prefer original Straw_kgha input data (and its uncertainty bounds) when present.
    if _row_has_col(row, "Straw_kgha"):
        straw_point = _row_float(row, "Straw_kgha", default=0.0)
        if straw_point > 0.0:
            return _resolve_uncertainty_bounds(row, "Straw_kgha", fallback_value=straw_point)

    # Otherwise derive straw uncertainty proportionally from yield uncertainty.
    lo_y, hi_y = _resolve_uncertainty_bounds(row, "Yield_kgha", fallback_value=_row_float(row, "Yield_kgha", default=0.0))

    crop_type = _infer_crop_type_for_row(row)
    factors = CROP_RESIDUE_WASTE_FACTORS
    ctype = crop_type or str(factors.get("default_crop_type", "cereal")).lower()
    ratio_map = factors.get("residue_to_yield_ratio_by_crop_type", {}) or {}
    rec_map = factors.get("recoverable_fraction_by_crop_type", {}) or {}
    ratio = float(ratio_map.get(ctype, ratio_map.get(str(factors.get("default_crop_type", "cereal")).lower(), 0.0)) or 0.0)
    recoverable = float(rec_map.get(ctype, rec_map.get(str(factors.get("default_crop_type", "cereal")).lower(), 1.0)) or 0.0)

    lo = max(0.0, float(lo_y) * ratio * recoverable)
    hi = max(0.0, float(hi_y) * ratio * recoverable)
    return min(lo, hi), max(lo, hi)


In [ ]:
# Optional manual overrides if XML comment != table column
MANUAL_COMMENT_TO_COLUMN = {
    # 'Fuel': 'Fuel_ha',
}

comment_to_col, mapping_df, unmatched_comments = build_comment_mapping(
    TEMPLATE_XML_PATH,
    df.columns.tolist(),
    manual_overrides=MANUAL_COMMENT_TO_COLUMN,
)

unmatched_summary = classify_unmatched_comments(unmatched_comments, df.columns.tolist())

print(f"Mapped comments (direct): {len(mapping_df)}")
print(f"Unmatched comments (total): {len(unmatched_comments)}")
print(f"Handled by legacy-style rules: {len(unmatched_summary['legacy_handled'])}")
print(f"Need extra columns/inputs for legacy rules: {len(unmatched_summary['needs_extra_inputs'])}")
print(f"Still unmatched after direct+legacy classification: {len(unmatched_summary['truly_unmatched'])}")
display(mapping_df)

if unmatched_summary['legacy_handled']:
    print('Rule-handled comment values (legacy-style logic):')
    for c in unmatched_summary['legacy_handled']:
        print('-', c)

if unmatched_summary['needs_extra_inputs']:
    print('Comments that need extra row inputs (e.g., Molecule_name_i/Dose_kgha_i or Transport_type_i):')
    for c in unmatched_summary['needs_extra_inputs']:
        print('-', c)

if unmatched_summary['truly_unmatched']:
    print('Still unmatched comment values:')
    for c in unmatched_summary['truly_unmatched']:
        print('-', c)


In [ ]:
def report_legacy_input_requirements(
    df: pd.DataFrame,
    out_csv: Path | None = None,
    timestamped: bool = False,
):
    cols = set(df.columns)

    groups = [
        {
            "rule": "Pesticide molecule emissions (c_PrimAir / soil_factor / c_PrimWater)",
            "required_any_prefixes": [],
            "required_all": ["Crop"],
            "recommended_any": ["Total_pesticides_kgha", "Insecticide_kgha", "Herbicide_kgha", "Fungicide_kgha"],
            "notes": "Primary path uses crop shares from inputs/02-05_espac_lci_coefficients.yml and partitions from inputs/03-05_dfe_factors.yml; indexed molecule columns are optional fallback.",
        },
        {
            "rule": "Transport mapping (Truck / Tractor)",
            "required_any_prefixes": [("Transport_type_", "Transport_tkm_")],
            "required_all": [],
            "notes": "Needs indexed transport type and tkm columns.",
        },
        {
            "rule": "Extra emissions / trace placeholders in template",
            "required_any_prefixes": [],
            "required_all": ["CH4", "PO4", "SOC_mean_rate_kgChayr", "Straw_kgha"],
            "optional_aliases": {"Ch": ["Ch", "Cr"]},
            "notes": "These are used only if precomputed values exist in the table.",
        },
        {
            "rule": "Packaging split by type",
            "required_any_prefixes": [],
            "required_all": ["Packaging_type1", "Packaging_type2"],
            "recommended_any": ["Packaging_type1_kgha", "Packaging_type2_kgha"],
            "notes": "Mass columns are needed to populate PP/PE/PET/LDPE/Kraft/Cardboard exchanges.",
        },
        {
            "rule": "Fuel conversion (Fuel -> MJ)",
            "required_any_prefixes": [],
            "required_all": ["Fuel_ha", "Fuel_type", "Fuel_unit"],
            "notes": "Legacy rule converts liters to MJ using a fixed factor when unit is L.",
        },
        {
            "rule": "Organic inputs aliases",
            "required_any_prefixes": [],
            "required_all": ["Manure_kgha", "Manure_type"],
            "recommended_any": ["Compost_kgha", "Compost_kg_ha", "Digestate_kgha", "Digestate_kg_ha"],
            "notes": "Compost/digestate use alias matching; manure placeholders depend on type.",
        },
        {
            "rule": "Seeds by crop type",
            "required_any_prefixes": [],
            "required_all": ["Seed_kgha"],
            "recommended_any": ["Custom_classification_3", "Category"],
            "notes": "Crop type is inferred from Custom_classification_3, falling back to Category.",
        },
        {
            "rule": "Yield/Area placeholder replacement",
            "required_any_prefixes": [],
            "required_all": ["Yield_kgha"],
            "recommended_any": ["Area_ha"],
            "notes": "Uses Yield_kgha for meanValue and inserts Area_ha into generalComment text.",
        },
    ]

    rows = []
    for g in groups:
        required_all = g.get("required_all", [])
        missing_required = [c for c in required_all if c not in cols]

        prefix_groups = g.get("required_any_prefixes", [])
        prefix_status_bits = []
        prefix_ok = True
        for pair in prefix_groups:
            bits = []
            pair_ok = True
            for pref in pair:
                present = any(str(c).startswith(pref) for c in cols)
                bits.append(f"{pref}*={'yes' if present else 'no'}")
                pair_ok = pair_ok and present
            prefix_status_bits.append(", ".join(bits))
            prefix_ok = prefix_ok and pair_ok

        recommended_any = g.get("recommended_any", [])
        missing_recommended = [c for c in recommended_any if c not in cols]

        alias_status = []
        for alias_name, options in g.get("optional_aliases", {}).items():
            present = [c for c in options if c in cols]
            alias_status.append(f"{alias_name} -> {present if present else 'missing'}")

        ready = (not missing_required) and prefix_ok
        rows.append({
            "rule": g["rule"],
            "ready": ready,
            "missing_required": ", ".join(missing_required) if missing_required else "",
            "prefix_checks": " | ".join(prefix_status_bits),
            "missing_recommended": ", ".join(missing_recommended) if missing_recommended else "",
            "alias_status": " | ".join(alias_status),
            "notes": g.get("notes", ""),
        })

    report_df = pd.DataFrame(rows)
    if out_csv is None:
        if timestamped:
            stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
            out_csv = PROJECT_DIR / f"reports/05_legacy_input_requirements_report_{stamp}.csv"
        else:
            out_csv = PROJECT_DIR / "reports/05_legacy_input_requirements_report.csv"
    elif timestamped:
        # If a base path is provided, append timestamp before suffix to avoid overwriting.
        out_csv = Path(out_csv)
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        out_csv = out_csv.with_name(f"{out_csv.stem}_{stamp}{out_csv.suffix}")

    print(f"Legacy rule input check across {len(cols)} columns")
    display(report_df)
    report_df.to_csv(out_csv, index=False)
    print(f"Saved diagnostics CSV: {out_csv}")

    not_ready = report_df.loc[~report_df["ready"], ["rule", "missing_required", "prefix_checks"]]
    if len(not_ready):
        print("Rules not fully activatable with current table columns:")
        for _, r in not_ready.iterrows():
            print("-", r["rule"])
            if r["missing_required"]:
                print("  missing required:", r["missing_required"])
            if r["prefix_checks"]:
                print("  prefix checks:", r["prefix_checks"])

    return report_df


# Set timestamped=True to keep a dated history of diagnostics reports
legacy_input_requirements_df = report_legacy_input_requirements(df, timestamped=False)


### Uncertainty Processing (new)
This notebook now reads uncertainty tables and writes uncertainty bounds into each XML exchange.
- It loads `<lci_table_stem>_uncertainty.csv` (or filtered fallback).
- For each exchange except reference product `exchange[@number="1"]`, it writes `uncertaintyType="3"`, plus `minValue` and `maxValue`.
- Bounds are taken from mapped table columns when available; otherwise they fallback to the exchange point value.


In [ ]:
ORGANIC_EXPLICIT_COLUMNS_BY_COMMENT = {
    "Cattle, solid": ("Manure_kgha",),
    "Cattle, liquid": ("Slurry_cattle_kgha",),
    "Swine, liquid": ("Slurry_swine_kgha",),
    "Poultry, droppings": ("Manure_poultry_kgha",),
    "Compost": ("Compost_kg_ha", "Compost_kgha"),
    "Compost_digestate": ("Compost_digestate_kgha",),
    "Digestate": ("Digestate_kg_ha", "Digestate_kgha"),
}

ORGANIC_AGGREGATE_COLUMN_BY_COMMENT = {
    "Cattle, solid": "Organic_estiercol_kgha",
    "Poultry, droppings": "Organic_estiercol_kgha",
    "Cattle, liquid": "Organic_liquido_kgha",
    "Swine, liquid": "Organic_liquido_kgha",
    "Digestate": "Organic_liquido_kgha",
    "Compost": "Organic_fermentado_kgha",
    "Compost_digestate": "Organic_fermentado_kgha",
}

ORGANIC_SPLIT_GROUPS = {
    "Organic_estiercol_kgha": ("Cattle, solid", "Poultry, droppings"),
    "Organic_liquido_kgha": ("Cattle, liquid", "Swine, liquid", "Digestate"),
    "Organic_fermentado_kgha": ("Compost", "Compost_digestate"),
}

DEFAULT_ORGANIC_SPLIT_SHARES = {
    "Cattle, solid": 0.90,
    "Poultry, droppings": 0.10,
    "Cattle, liquid": 0.08,
    "Swine, liquid": 0.88,
    "Digestate": 0.04,
    "Compost": 0.50,
    "Compost_digestate": 0.50,
}

ORGANIC_SPLIT_SHARES = dict(DEFAULT_ORGANIC_SPLIT_SHARES)
ORGANIC_FERMENTADO_CAP_KGHA = 20000.0
for _k, _v in ((DFE_ASSUMPTIONS.get("organic_split_shares", {}) or {}).items() if isinstance(DFE_ASSUMPTIONS, dict) else []):
    try:
        ORGANIC_SPLIT_SHARES[str(_k)] = float(_v)
    except Exception:
        pass

def _resolve_sowing_share_and_bounds_per_ha(row: pd.Series) -> tuple[float, float, float]:
    area_ha = _row_float(row, "Area_ha", default=0.0)
    if area_ha <= 0.0:
        return 0.0, 0.0, 0.0

    sow_col = None
    sow_candidates = globals().get("SOWING_AREA_COLUMN_CANDIDATES", (
        "Sowing_mechanized_area_ha",
        "Sowing_mechanised_area_ha",
        "Sowing_area_ha",
        "Seeded_area_ha",
        "Seeding_area_ha",
    ))
    for c in sow_candidates:
        if _row_has_col(row, c):
            sow_col = c
            break

    sow_area = _row_float(row, sow_col, default=0.0) if sow_col else 0.0
    mean_share = max(0.0, float(sow_area) / float(area_ha))

    if sow_col:
        lo_area, hi_area = _resolve_uncertainty_bounds(row, sow_col, fallback_value=sow_area)
    else:
        lo_area, hi_area = sow_area, sow_area

    lo_share = max(0.0, float(lo_area) / float(area_ha))
    hi_share = max(0.0, float(hi_area) / float(area_ha))
    return mean_share, min(lo_share, hi_share), max(lo_share, hi_share)


def _resolve_organic_exchange_value_and_bounds(row: pd.Series, comment: str):
    total_org = _row_float(row, "Total_fert_org_kgha", default=np.nan)
    est = _row_float(row, "Organic_estiercol_kgha", default=0.0)
    liq = _row_float(row, "Organic_liquido_kgha", default=0.0)

    # Guardrail at XML stage: enforce same plausibility cap in case upstream table was generated before cap logic.
    fer_raw = _row_float(row, "Organic_fermentado_kgha", default=np.nan)
    if pd.notna(fer_raw):
        fer_capped = min(max(0.0, float(fer_raw)), ORGANIC_FERMENTADO_CAP_KGHA)
        if pd.isna(total_org):
            total_org = max(0.0, est) + max(0.0, liq) + fer_capped
        else:
            total_org = min(max(0.0, float(total_org)), max(0.0, est) + max(0.0, liq) + fer_capped)

    def _coherent_cap_for_comment(value: float) -> float:
        if pd.isna(total_org):
            return max(0.0, float(value))
        v = max(0.0, float(value))
        if comment in {"Compost", "Compost_digestate"}:
            fer_cap = max(0.0, float(total_org) - max(0.0, est) - max(0.0, liq))
            return min(v, fer_cap)
        return min(v, max(0.0, float(total_org)))

    # Prefer explicit per-exchange columns when present.
    for col in ORGANIC_EXPLICIT_COLUMNS_BY_COMMENT.get(comment, ()):
        if _row_has_col(row, col):
            raw_val = _row_float(row, col, default=0.0)
            if comment in {"Compost", "Compost_digestate"}:
                raw_val = min(max(0.0, float(raw_val)), ORGANIC_FERMENTADO_CAP_KGHA)
            mean_val = _coherent_cap_for_comment(raw_val)
            lo, hi = _resolve_uncertainty_bounds(row, col, fallback_value=mean_val)
            if comment in {"Compost", "Compost_digestate"}:
                lo = min(max(0.0, float(lo)), ORGANIC_FERMENTADO_CAP_KGHA)
                hi = min(max(0.0, float(hi)), ORGANIC_FERMENTADO_CAP_KGHA)
            lo = min(float(lo), mean_val)
            hi = max(float(hi), mean_val)
            return float(mean_val), float(lo), float(hi)

    # Fallback to aggregate organic pools split by predefined shares.
    agg_col = ORGANIC_AGGREGATE_COLUMN_BY_COMMENT.get(comment)
    if agg_col and _row_has_col(row, agg_col):
        total_mean = _row_float(row, agg_col, default=0.0)
        if agg_col == "Organic_fermentado_kgha":
            total_mean = min(max(0.0, float(total_mean)), ORGANIC_FERMENTADO_CAP_KGHA)
        if agg_col == "Organic_fermentado_kgha" and not pd.isna(total_org):
            total_mean = min(max(0.0, total_mean), max(0.0, float(total_org) - max(0.0, est) - max(0.0, liq)))
        share = float(ORGANIC_SPLIT_SHARES.get(comment, 0.0) or 0.0)
        mean_val = float(max(0.0, total_mean) * share)
        lo_total, hi_total = _resolve_uncertainty_bounds(row, agg_col, fallback_value=total_mean)
        if agg_col == "Organic_fermentado_kgha":
            lo_total = min(max(0.0, float(lo_total)), ORGANIC_FERMENTADO_CAP_KGHA)
            hi_total = min(max(0.0, float(hi_total)), ORGANIC_FERMENTADO_CAP_KGHA)
        return mean_val, float(max(0.0, lo_total) * share), float(max(0.0, hi_total) * share)

    return None


def _validate_and_correct_uncertainty_bounds(mean_val: float, min_val: float, max_val: float, sample_count: float = np.nan) -> tuple:
    """Ensure coherent bounds and force non-point intervals when sample_count > 1."""
    if min_val > mean_val:
        min_val = mean_val
    if max_val < mean_val:
        max_val = mean_val
    if min_val > max_val:
        min_val, max_val = max_val, min_val

    if pd.notna(sample_count) and float(sample_count) > 1 and float(min_val) == float(max_val):
        span = max(abs(float(mean_val)) * 0.01, 1e-12)
        if float(mean_val) == 0.0:
            min_val, max_val = 0.0, 1e-12
        else:
            min_val = max(0.0, float(mean_val) - span)
            max_val = float(mean_val) + span

    return float(min_val), float(max_val)


def _qname(local_name: str, ns: dict) -> str:
    if ns and ns.get("spold"):
        return f"{{{ns['spold']}}}{local_name}"
    return local_name


def _find_or_create(parent, child_name: str, ns: dict):
    child = parent.find(f"spold:{child_name}", ns) if ns else parent.find(child_name)
    if child is None:
        child = ET.SubElement(parent, _qname(child_name, ns))
    return child


EXCHANGE_NAMES_TO_REMOVE = {
    "transport, cami?, mitjana espanyola de vehicles pesants (ES) transport de 14 a 32 i +32 tn / 2019",
}


def _remove_exchanges_by_name(root, ns: dict, names_to_remove: set[str]):
    if not names_to_remove:
        return 0

    if ns:
        flow = root.find('.//spold:flowData', ns)
    else:
        flow = root.find('.//flowData')
    if flow is None:
        return 0

    removed = 0
    for ex in list(flow):
        tag = ex.tag.split('}', 1)[-1] if isinstance(ex.tag, str) else str(ex.tag)
        if tag != 'exchange':
            continue
        ex_name = str(ex.attrib.get('name', '')).strip()
        if ex_name in names_to_remove:
            flow.remove(ex)
            removed += 1

    # Keep numbering contiguous after deletions.
    n = 1
    for ex in list(flow):
        tag = ex.tag.split('}', 1)[-1] if isinstance(ex.tag, str) else str(ex.tag)
        if tag == 'exchange':
            ex.attrib['number'] = str(n)
            n += 1

    return removed

SIMAPRO_CLASSIFICATION = ESPAC_COEFFS.get("simapro_classification", {}) or {}
SIMAPRO_AGRI_CATEGORY = str(SIMAPRO_CLASSIFICATION.get("category", "Agricultural"))
SIMAPRO_AGRI_LOCAL_CATEGORY = str(SIMAPRO_CLASSIFICATION.get("localCategory", SIMAPRO_AGRI_CATEGORY))
SIMAPRO_AGRI_SUBCATEGORY = str(SIMAPRO_CLASSIFICATION.get("subCategory", "ECUADOR"))
SIMAPRO_AGRI_LOCAL_SUBCATEGORY = str(SIMAPRO_CLASSIFICATION.get("localSubCategory", SIMAPRO_AGRI_SUBCATEGORY))

REFERENCE_PRODUCT_LOCATION = "EC"

SIMAPRO_SCHEMA_LOCATION = (
    "http://www.EcoInvent.org/EcoSpold01 "
    "http://www.EcoInvent.org/EcoSpold01..\\..\\EcoSpold01Dataset.xsd "
    "https://github.com/brightway-lca/pyecospold/tree/main/pyecospold/schemas/v1"
)


def _reorder_modelling_and_validation_children(modelling, representativeness, validation, source):
    children = list(modelling)
    ordered = []
    for node in (representativeness, validation, source):
        if node is not None and node in children and node not in ordered:
            ordered.append(node)
    for node in children:
        if node not in ordered:
            ordered.append(node)
    for node in list(modelling):
        modelling.remove(node)
    for node in ordered:
        modelling.append(node)


def _prepare_tree_for_simapro_serialization(tree, ns: dict):
    if ns and ns.get("spold"):
        ET.register_namespace("", ns["spold"])
    ET.register_namespace("xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root = tree.getroot()
    root.attrib["{http://www.w3.org/2001/XMLSchema-instance}schemaLocation"] = SIMAPRO_SCHEMA_LOCATION


def _ensure_simapro_validation_metadata(root, ns: dict, validation_meta: dict, generation_date: str):
    meta = root.find('.//spold:metaInformation', ns) if ns else root.find('.//metaInformation')
    if meta is None:
        return
    modelling = _find_or_create(meta, 'modellingAndValidation', ns)
    representativeness = _find_or_create(modelling, 'representativeness', ns)
    source = _find_or_create(modelling, 'source', ns)
    validation = _find_or_create(modelling, 'validation', ns)

    def _set_text(parent, child_name, value):
        child = _find_or_create(parent, child_name, ns)
        child.text = "" if value is None else str(value)

    _set_text(validation, 'reviewStatus', validation_meta.get('reviewStatus', 'Final'))
    _set_text(validation, 'reviewer', validation_meta.get('reviewer', ''))
    _set_text(validation, 'reviewDate', generation_date)

    validator_meta = validation_meta.get('proofReadingValidator', {}) or {}
    proof = _find_or_create(validation, 'proofReadingValidator', ns)
    _set_text(proof, 'name', validator_meta.get('name', ''))
    _set_text(proof, 'email', validator_meta.get('email', ''))
    _set_text(proof, 'companyCode', validator_meta.get('companyCode', ''))
    _set_text(proof, 'proofReadingDate', generation_date)

    _reorder_modelling_and_validation_children(modelling, representativeness, validation, source)

def _apply_exchange_location_unit_rules(ex, geography_location: str):
    # SimaPro requires exchange #1 location to match dataset geography.
    number_raw = str(ex.attrib.get("number", "")).strip()
    try:
        number = int(number_raw)
    except Exception:
        number = None

    if number == 1:
        ex.attrib["location"] = REFERENCE_PRODUCT_LOCATION
    else:
        ex.attrib["location"] = "RER"

    # Keep template-defined units; do not coerce exchange units here.


def _ensure_exchange_group_tag(ex, ns: dict):
    def _lname(tag):
        return tag.split("}", 1)[1] if isinstance(tag, str) and tag.startswith("{") and "}" in tag else str(tag)

    has_group = any(_lname(child.tag) in {"inputGroup", "outputGroup"} for child in list(ex))
    if has_group:
        return

    # Elementary/resource flows use outputGroup=4; technosphere defaults to inputGroup=5.
    if "category" in ex.attrib:
        group = ET.SubElement(ex, _qname("outputGroup", ns))
        group.text = "4"
    else:
        group = ET.SubElement(ex, _qname("inputGroup", ns))
        group.text = "5"

def render_xml_for_row(template_xml_path: Path, row: pd.Series, comment_to_col: dict, na_default=0.0, generation_date=None, summary_token="province"):
    tree, root, ns = get_root_and_ns(template_xml_path)
    generation_date = generation_date or date.today().isoformat()
    _ensure_simapro_validation_metadata(root, ns, XML_VALIDATION_METADATA, generation_date)
    row_irrigation_source = _resolve_irrigation_source_for_row(row, IRRIGATION_WATER_SOURCE)

    _remove_exchanges_by_name(root, ns, EXCHANGE_NAMES_TO_REMOVE)

    # Fill exchange meanValue using (1) direct comment->column mapping and (2) legacy-style rule logic
    if ns:
        geo = root.find('.//spold:geography', ns)
    else:
        geo = root.find('.//geography')
    if geo is not None:
        geo.attrib["location"] = REFERENCE_PRODUCT_LOCATION
    geography_location = REFERENCE_PRODUCT_LOCATION
    land_use_name_map = _land_use_name_mapping_for_row(row)

    for ex in get_exchange_elements(root, ns):
        ex_name_lc = str(ex.attrib.get("name", "")).strip().lower()
        if ex_name_lc in land_use_name_map:
            ex.attrib["name"] = land_use_name_map[ex_name_lc]
            ex_name_lc = str(ex.attrib.get("name", "")).strip().lower()

        _apply_exchange_location_unit_rules(ex, geography_location)
        _ensure_exchange_group_tag(ex, ns)
        comment = ex.attrib.get("generalComment", "").strip()

        # Set unit to m2 for Occupation/Transformation exchanges if not already set
        ex_name_lower = str(ex.attrib.get("name", "")).lower()
        if any(keyword in ex_name_lower for keyword in ["occupation", "transformation"]):
            if "m2" not in ex.attrib.get("unit", ""):
                ex.attrib["unit"] = "m2"

        if comment and comment in comment_to_col:
            col = comment_to_col[comment]
            ex.attrib["meanValue"] = str(to_numeric_or_default(row.get(col), default=na_default))

        rule_value = apply_legacy_rule_to_exchange(ex, row)
        if rule_value is not None:
            ex.attrib["meanValue"] = str(to_numeric_or_default(rule_value, default=na_default))

        # Resolve uncertainty bounds from direct mapping, legacy alias mapping, or fallback to point value.
        mean_now = to_numeric_or_default(ex.attrib.get("meanValue", na_default), default=na_default)
        source_col = None
        if "market for sowing" in ex_name_lower:
            sow_share, sow_min, sow_max = _resolve_sowing_share_and_bounds_per_ha(row)
            ex.attrib["unit"] = "ha"
            ex.attrib["meanValue"] = str(sow_share)
            min_v, max_v = float(sow_min), float(sow_max)
        elif comment == "Fuel":
            min_v, max_v = _resolve_fuel_uncertainty_bounds(
                row,
                ex.attrib.get("name", ""),
                fallback_value=mean_now,
            )
        elif comment == "Straw_kgha":
            min_v, max_v = _resolve_straw_uncertainty_bounds(row, fallback_value=mean_now)
        elif comment in {"c_PrimAir", "c_PrimWater", "soil_factor"}:
            min_v, max_v = _resolve_pesticide_emission_uncertainty_bounds(
                row,
                ex.attrib.get("name", ""),
                comment,
                fallback_value=mean_now,
            )
        elif comment in ORGANIC_AGGREGATE_COLUMN_BY_COMMENT:
            organic_values = _resolve_organic_exchange_value_and_bounds(row, comment)
            if organic_values is not None:
                ex.attrib["unit"] = "kg"
                organic_mean, organic_min, organic_max = organic_values
                ex.attrib["meanValue"] = str(organic_mean)
                min_v, max_v = float(organic_min), float(organic_max)
            else:
                min_v, max_v = float(mean_now), float(mean_now)
        elif comment == "SOC_mean_rate_kgChayr":
            soc_mean, soc_min, soc_max = _resolve_soc_value_and_bounds(row)
            ex.attrib["meanValue"] = str(soc_mean)
            min_v, max_v = float(soc_min), float(soc_max)
        else:
            if comment and comment in comment_to_col:
                source_col = comment_to_col[comment]
            else:
                source_col = _legacy_uncertainty_alias_col(comment)

            if source_col:
                min_v, max_v = _resolve_uncertainty_bounds(row, source_col, fallback_value=mean_now)
            else:
                min_v, max_v = float(mean_now), float(mean_now)

        ex.attrib["uncertaintyType"] = "3"
        mean_val = float(ex.attrib.get("meanValue", 0))
        sample_count = _row_float(row, "count", default=np.nan)
        min_v, max_v = _validate_and_correct_uncertainty_bounds(mean_val, float(min_v), float(max_v), sample_count=sample_count)
        ex.attrib["minValue"] = str(min_v)
        ex.attrib["maxValue"] = str(max_v)

        # Route irrigation water by source. Prefer source-specific columns whose sum equals Irrig_m3;
        # fall back to the row/default source when older tables only have the total column.
        if comment == "Irrig_m3":
            irrig_source = _get_irrigation_exchange_source(ex.attrib.get("name", ""))
            if irrig_source:
                source_col = IRRIGATION_SOURCE_VALUE_COLUMNS.get(irrig_source)
                total_irrig = _row_float(row, "Irrig_m3", default=0.0)
                total_min, total_max = _resolve_uncertainty_bounds(row, "Irrig_m3", fallback_value=total_irrig)
                split_total = 0.0
                for _src, _col in IRRIGATION_SOURCE_VALUE_COLUMNS.items():
                    if _col in row.index:
                        split_total += max(0.0, to_numeric_or_default(row.get(_col), default=0.0))

                if source_col and source_col in row.index and split_total > 0.0:
                    source_value = max(0.0, to_numeric_or_default(row.get(source_col), default=0.0))
                    source_share = source_value / split_total if split_total > 0.0 else 0.0
                    cmin_src, cmax_src = _resolve_uncertainty_columns(row, source_col)
                    has_source_bounds = bool(cmin_src and cmax_src and cmin_src in row.index and cmax_src in row.index)
                    if has_source_bounds:
                        source_min, source_max = _resolve_uncertainty_bounds(row, source_col, fallback_value=source_value)
                    else:
                        # No source-specific uncertainty columns: allocate total irrigation bounds by source share.
                        source_min = max(0.0, float(total_min) * float(source_share))
                        source_max = max(0.0, float(total_max) * float(source_share))
                    ex.attrib["meanValue"] = str(source_value)
                    ex.attrib["minValue"] = str(source_min)
                    ex.attrib["maxValue"] = str(source_max)
                else:
                    selected = _norm_token(row_irrigation_source) or _norm_token(IRRIGATION_WATER_SOURCE)
                    if _norm_token(irrig_source) == selected and total_irrig > 0.0:
                        ex.attrib["meanValue"] = str(total_irrig)
                        ex.attrib["minValue"] = str(total_min)
                        ex.attrib["maxValue"] = str(total_max)
                    else:
                        ex.attrib["meanValue"] = "0.0"
                        ex.attrib["minValue"] = "0.0"
                        ex.attrib["maxValue"] = "0.0"
        # Set land occupation/transformation flows to 1 ha in native units (m2a/m2).
        ex_name_lc = str(ex.attrib.get("name", "")).strip().lower()
        if ex_name_lc in LAND_NORMALIZED_FLOW_NAMES:
            ex.attrib["meanValue"] = "10000.0"
            ex.attrib["minValue"] = "10000.0"
            ex.attrib["maxValue"] = "10000.0"


    # Optional process naming from row fields
    proc_name = _process_name_for_row(row, summary_token)

    if ns:
        ref = root.find('.//spold:referenceFunction', ns)
        ex1 = root.find('.//spold:flowData/spold:exchange[@number="1"]', ns)
        tech = root.find('.//spold:technology', ns)
    else:
        ref = root.find('.//referenceFunction')
        ex1 = root.find('.//flowData/exchange[@number="1"]')
        tech = root.find('.//technology')

    if ref is not None:
        ref.attrib['name'] = proc_name
        ref.attrib['localName'] = proc_name
        ref.attrib['category'] = SIMAPRO_AGRI_CATEGORY
        ref.attrib['localCategory'] = SIMAPRO_AGRI_LOCAL_CATEGORY
        ref.attrib['subCategory'] = SIMAPRO_AGRI_SUBCATEGORY
        ref.attrib['localSubCategory'] = SIMAPRO_AGRI_LOCAL_SUBCATEGORY
        if 'Yield_kgha' in row.index:
            ref.attrib['amount'] = str(to_numeric_or_default(row.get('Yield_kgha'), default=na_default))

    ex1_comment = ""
    if ex1 is not None:
        ex1.attrib['name'] = proc_name
        ex1_comment = str(ex1.attrib.get('generalComment', '')).strip()
        n_raw = to_numeric_or_default(row.get('count', np.nan), default=np.nan)
        n_txt = f"n = {int(round(float(n_raw)))}" if np.isfinite(n_raw) else "n = ?"
        ex1_comment_clean = re.sub(r"\s*\|\s*n\s*=\s*\?*\d+\s*$", "", ex1_comment).strip()
        if ex1_comment_clean:
            ex1_comment = f"{ex1_comment_clean} | {n_txt}"
        else:
            ex1_comment = n_txt
        ex1.attrib['generalComment'] = ex1_comment
        # SimaPro compatibility: keep reference product exchange free of uncertainty bounds.
        for attr in ('uncertaintyType', 'minValue', 'maxValue'):
            ex1.attrib.pop(attr, None)
        # Fallback for SimaPro UIs that hide reference exchange comments.
        if ex1_comment and tech is not None:
            tech.attrib['text'] = ex1_comment

    # Keep the same note on land occupation exchange as on reference exchange.
    if ex1_comment:
        for ex in get_exchange_elements(root, ns):
            ex_name_lc = str(ex.attrib.get('name', '')).strip().lower()
            if ex_name_lc in {'occupation, annual crop', 'occupation, annual crop, ec', 'occupation, permanent crop', 'occupation, permanent crop, ec'}:
                ex.attrib['generalComment'] = ex1_comment
                break

    _prepare_tree_for_simapro_serialization(tree, ns)
    return tree


def export_xmls_from_table(df: pd.DataFrame, template_xml_path: Path, out_dir: Path, comment_to_col: dict,
                           na_default=0.0, limit=None, summary_token="province"):
    out_dir.mkdir(parents=True, exist_ok=True)
    work = df.copy()
    if limit is not None:
        work = work.head(int(limit)).copy()

    written = []
    skipped = []
    generation_date = date.today().isoformat()
    def _reference_output_mean_value_from_tree(tree: ET.ElementTree):
        root = tree.getroot()

        def _lname(tag):
            return str(tag).split('}', 1)[-1] if isinstance(tag, str) else str(tag)

        ex_candidates = [e for e in root.iter() if _lname(e.tag) == 'exchange']
        if not ex_candidates:
            return np.nan

        ref_group = []
        for ex in ex_candidates:
            og_text = str(ex.attrib.get('outputGroup', '')).strip()
            if not og_text:
                for ch in list(ex):
                    if _lname(ch.tag) == 'outputGroup':
                        og_text = (ch.text or '').strip()
                        break
            og_num = to_numeric_or_default(og_text, default=np.nan)
            if np.isfinite(og_num) and int(og_num) == 0:
                ref_group.append(ex)

        if not ref_group:
            # Fallback: some templates may miss outputGroup on the reference exchange.
            for ex in ex_candidates:
                if str(ex.attrib.get('number', '')).strip() == '1':
                    return to_numeric_or_default(ex.attrib.get('meanValue', np.nan), default=np.nan)
            return np.nan

        for ex in ref_group:
            if str(ex.attrib.get('number', '')).strip() == '1':
                return to_numeric_or_default(ex.attrib.get('meanValue', np.nan), default=np.nan)

        vals = []
        for ex in ref_group:
            mv = to_numeric_or_default(ex.attrib.get('meanValue', np.nan), default=np.nan)
            if np.isfinite(mv):
                vals.append(float(mv))
        return max(vals) if vals else np.nan
    for i, row in work.reset_index(drop=True).iterrows():
        tree = render_xml_for_row(
            template_xml_path,
            row,
            comment_to_col,
            na_default=na_default,
            generation_date=generation_date,
            summary_token=summary_token,
        )
        ref_mean = _reference_output_mean_value_from_tree(tree)
        if (not np.isfinite(ref_mean)) or (float(ref_mean) <= 0.0):
            skipped.append({
                'row_number': int(i + 1),
                'Crop': str(row.get('Crop', '')),
                'Category': str(row.get('Category', '')),
                'reference_output_meanValue': ref_mean,
                'reason': 'reference output (outputGroup=0) meanValue is non-positive or invalid',
            })
            continue
        fname = _filename_for_row(row, summary_token, row_number=i+1)
        out_path = out_dir / fname
        tree.write(out_path, encoding='UTF-8', xml_declaration=True)
        written.append(out_path)

    globals()['LAST_XML_EXPORT_SKIPPED'] = skipped
    return written



In [ ]:
def _resolve_soc_value_and_bounds(row: pd.Series) -> tuple[float, float, float]:
    """Resolve SOC point value and uncertainty bounds with robust fallbacks."""
    base_col = "SOC_mean_rate_kgChayr"
    mean_val = _row_float(row, base_col, default=0.0)

    # Preferred canonical uncertainty columns.
    lo, hi = _resolve_uncertainty_bounds(row, base_col, fallback_value=mean_val)

    # Fallback for malformed columns observed in some exports.
    weird_lo = _to_uncertainty_float(row.get("SOC_mean_rate_kgChayr__maxValue__minValue", np.nan))
    weird_hi = _to_uncertainty_float(row.get("SOC_mean_rate_kgChayr__minValue__maxValue", np.nan))
    if weird_lo is not None and weird_hi is not None:
        lo = min(float(weird_lo), float(weird_hi))
        hi = max(float(weird_lo), float(weird_hi))

    # Ensure valid ordering in all cases.
    lo = min(float(lo), float(hi))
    hi = max(float(lo), float(hi))
    return float(mean_val), float(lo), float(hi)

In [ ]:
# Ensure OTROS and aggregate crop placeholders use inherited actual-crop fields when available.

OTROS_PLACEHOLDER_CROPS = {"OTROS PERMANENTES", "OTROS TRANSITORIOS"}
GENERIC_CROP_PLACEHOLDERS = OTROS_PLACEHOLDER_CROPS | {"ALL", "(ALL CROPS IN GROUP)", "ALL CROPS"}
OTROS_ACTUAL_CROP_COLUMN_CANDIDATES = [
    "Actual_crop_name",
    "Actual_crop",
    "Selected_crop",
    "Selected_subcrop",
    "Subcrop",
    "Sub_crop",
    "Crop_detail",
    "Detailed_crop",
    "Specific_crop",
    "Inherited_crop_name",
    "Original_crop_name",
    "Source_crop_name",
    "rc_clacul",
    "ct_codcultiv1_int",
    "ct_codcultiv2_int",
]
OTROS_ACTUAL_CROP_META_KEYS = [
    # Prefer specific subcrop first; selected_crop is often just "All".
    "selected_subcrop",
    "otros_crop_filter",
    "actual_crop_name",
    "selected_crop",
    "otros_crop",
]


def _parse_first_candidate_label(raw) -> str:
    txt = str(raw or "").strip()
    if not txt or txt.lower() == "nan":
        return ""

    invalid_tokens = {"NAN", "NONE", "UNKNOWN", "(UNKNOWN)", "ALL", "(ALL)", "ALL CROPS"}

    # Handle delimited values by keeping the first informative token.
    for sep in ["|", ";", "/", ","]:
        if sep in txt:
            parts = [p.strip() for p in txt.split(sep) if p and p.strip()]
            for p in parts:
                up = p.upper()
                if up not in GENERIC_CROP_PLACEHOLDERS and up not in invalid_tokens:
                    return p

    up = txt.upper()
    if up in GENERIC_CROP_PLACEHOLDERS or up in invalid_tokens:
        return ""
    return txt


def _summary_meta_selected_crop_name() -> str:
    meta = globals().get("summary_meta", None)
    if not isinstance(meta, dict):
        return ""
    for key in OTROS_ACTUAL_CROP_META_KEYS:
        if key in meta:
            candidate = _parse_first_candidate_label(meta.get(key))
            if candidate:
                return candidate
    return ""


def _resolved_crop_label_for_row(row: pd.Series) -> str:
    crop_raw = _clean_display_part(row.get("Crop", ""))
    crop_up = str(crop_raw).strip().upper()
    if crop_up not in GENERIC_CROP_PLACEHOLDERS:
        return crop_raw

    for col in OTROS_ACTUAL_CROP_COLUMN_CANDIDATES:
        if col in row.index and not pd.isna(row.get(col)):
            candidate = _parse_first_candidate_label(row.get(col))
            if candidate:
                return candidate

    # Fallback to metadata emitted by upstream summary notebooks, if present.
    meta_crop = _summary_meta_selected_crop_name()
    if meta_crop:
        return meta_crop

    # Keep legacy value if no inherited detail field is available.
    return crop_raw


def _display_name_and_slug_parts_for_row(row: pd.Series, summary_token: str, row_number=None):
    display_parts = []
    slug_parts = [f"{int(row_number):05d}"] if row_number is not None else []

    token = str(summary_token or "").strip().lower()
    if token == "crop_group_national":
        group, category = _strategy8_group_and_system_for_row(row)
        if group:
            display_parts.append(group)
            slug_parts.append(f"group_{safe_name(group).lower()}")

        display_parts.append(f"aggregation: {_aggregation_display(summary_token)}")
        return display_parts, slug_parts

    ctx = _strategy_context_for_row(row, summary_token)
    if ctx:
        display_parts.append(f"{ctx['label']}: {ctx['value']}")
        slug_parts.append(f"{ctx['slug_label']}_{safe_name(ctx['value']).lower()}")

    crop = _resolved_crop_label_for_row(row)
    if crop and not _is_confounded_placeholder(crop):
        display_parts.append(f"crop: {crop}")
        slug_parts.append(f"crop_{safe_name(crop).lower()}")

    category = _clean_display_part(row.get("Category", ""))
    ctx_value_lc = str((ctx or {}).get("value", "")).strip().lower()
    if category and category.lower() != ctx_value_lc:
        display_parts.append(f"category: {category}")
        slug_parts.append(f"category_{safe_name(category).lower()}")

    region = _clean_display_part(row.get("Region", ""))
    if region and not _is_confounded_placeholder(region):
        display_parts.append(f"region: {region}")
        slug_parts.append(f"region_{safe_name(region).lower()}")

    province = _clean_display_part(row.get("Province", ""))
    if province and not _is_confounded_placeholder(province):
        display_parts.append(f"province: {province}")
        slug_parts.append(f"province_{safe_name(province).lower()}")

    display_parts.append(f"aggregation: {_aggregation_display(summary_token)}")
    return display_parts, slug_parts


def _process_name_for_row(row: pd.Series, summary_token: str) -> str:
    display_parts, _ = _display_name_and_slug_parts_for_row(row, summary_token)
    return " | ".join(display_parts)[:255] if display_parts else "record"


def _filename_for_row(row: pd.Series, summary_token: str, row_number=None) -> str:
    _, slug_parts = _display_name_and_slug_parts_for_row(row, summary_token, row_number=row_number)
    return safe_name(*slug_parts) + ".xml"


print("OTROS naming override loaded: filenames and process labels now use inherited actual crop fields when available.")

In [ ]:
# Configure export
NA_TO_ZERO_DEFAULT = 0.0  # Set to None and adapt code if you want to preserve missing values
LIMIT = 100  # test cap: export at most 20 rows

for old_xml in OUTPUT_DIR.glob("*.xml"):
    old_xml.unlink()

written = export_xmls_from_table(
    df=df,
    template_xml_path=TEMPLATE_XML_PATH,
    out_dir=OUTPUT_DIR,
    comment_to_col=comment_to_col,
    na_default=NA_TO_ZERO_DEFAULT,
    limit=LIMIT,
    summary_token=SUMMARY_TOKEN,
)

print(f"XML files written: {len(written)}")
print('First files:')
for p in written[:5]:
    print('-', p)

skipped = globals().get('LAST_XML_EXPORT_SKIPPED', [])
if skipped:
    print(f"Warning: skipped {len(skipped)} rows with non-positive/invalid reference flow (not exported to XML).")
    display(pd.DataFrame(skipped))
else:
    print("No rows skipped for reference-flow validation.")


if written:
    xml_run_id = new_run_id('05_xml_crops')
    main_snapshot = make_snapshot_copy(LCI_TABLE_PATH, xml_run_id)
    unc_snapshot = make_snapshot_copy(UNCERTAINTY_TABLE_PATH if UNCERTAINTY_TABLE_PATH.exists() else LCI_TABLE_PATH.with_name(f"{LCI_TABLE_PATH.stem}_uncertainty.csv"), xml_run_id) if (UNCERTAINTY_TABLE_PATH.exists() or LCI_TABLE_PATH.with_name(f"{LCI_TABLE_PATH.stem}_uncertainty.csv").exists()) else main_snapshot
    rec = build_manifest_record(
        run_id=xml_run_id,
        domain='crops',
        summary_token=SUMMARY_TOKEN,
        pipeline_stage='05_xml',
        source_main_csv=main_snapshot,
        source_unc_csv=unc_snapshot,
        main_df=df,
        unc_df=df[[c for c in df.columns if c.endswith('__minValue') or c.endswith('__maxValue')]].copy() if isinstance(df, pd.DataFrame) else pd.DataFrame(),
        filters_meta=summary_meta if isinstance(summary_meta, dict) else {},
        upstream_run_id=str((summary_meta or {}).get('run_id', '') or None),
        extra={'xml_output_dir': str(OUTPUT_DIR.resolve()), 'xml_files_written': int(len(written)), 'otros_expansion_source': 'db'},
    )
    mpath = append_manifest_record(PROJECT_DIR, rec)
    print(f"Manifest updated: {mpath} (run_id={xml_run_id})")


In [ ]:
# Optional: inspect one generated XML quickly
if written:
    sample = written[0]
    print(sample)
    with open(sample, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 200:
                break
            print(line.rstrip())
